In [ ]:
# ==============================================================================
# 1. LIBRARY IMPORTS
# ==============================================================================
import re
import warnings
import numpy as np
import pandas as pd
import optuna

# Specific imports for data loading and modeling
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Define a constant random state for reproducibility
RANDOM_STATE = 42


# ==============================================================================
# 2. CUSTOM ALGORITHM DEFINITION
# ==============================================================================
def softmax(F):
    """Compute row-wise softmax."""
    expF = np.exp(F - np.max(F, axis=1, keepdims=True))
    return expF / np.sum(expF, axis=1, keepdims=True)

class CustomMulticlassGradientBoostingClassifier:
    """
    Custom multiclass Gradient Boosting classifier with LDA.
    """
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.estimators = []
        self.lda_transforms = []
        self.initial_logit = None
        self.classes_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()

        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        one_hot_y = np.eye(n_classes)[np.searchsorted(self.classes_, y)]
        
        prior = np.mean(one_hot_y, axis=0)
        prior = np.clip(prior, 1e-5, 1 - 1e-5)
        self.initial_logit = np.log(prior)
        
        F = np.tile(self.initial_logit, (n_samples, 1))
        
        for m in range(self.n_estimators):
            if m == 0:
                lda = LinearDiscriminantAnalysis(n_components=None)
                X_lda = lda.fit_transform(X, y)
            else:
                p = softmax(F)
                residuals = one_hot_y - p
                labels = np.argmax(residuals, axis=1)
                lda = LinearDiscriminantAnalysis(n_components=None)
                X_lda = lda.fit_transform(X, labels)
            
            self.lda_transforms.append(lda)
            
            p = softmax(F)
            residuals = one_hot_y - p
            
            estimators_m = []
            for k in range(n_classes):
                reg = DecisionTreeRegressor(max_depth=self.max_depth, random_state=RANDOM_STATE)
                reg.fit(X_lda, residuals[:, k])
                estimators_m.append(reg)
                update = reg.predict(X_lda)
                F[:, k] += self.learning_rate * update
            
            self.estimators.append(estimators_m)
        return self

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()

        n_samples = X.shape[0]
        F = np.tile(self.initial_logit, (n_samples, 1))
        for lda, estimators_m in zip(self.lda_transforms, self.estimators):
            X_lda = lda.transform(X)
            for k, reg in enumerate(estimators_m):
                F[:, k] += self.learning_rate * reg.predict(X_lda)
        return softmax(F)

    def predict(self, X):
        proba = self.predict_proba(X)
        class_indices = np.argmax(proba, axis=1)
        return self.classes_[class_indices]
        
    def get_params(self, deep=True):
        return {
            'n_estimators': self.n_estimators,
            'learning_rate': self.learning_rate,
            'max_depth': self.max_depth
        }


# ==============================================================================
# 3. DATA LOADING AND PREPROCESSING
# ==============================================================================
print("Start: Loading and preparing data...")

try:
    data = pd.read_csv("Data/train.csv", index_col="id")
except FileNotFoundError:
    print("Error: 'Data/train.csv' not found. Make sure the path is correct.")
    exit()

y = data["rainfall"]
X = data.drop(columns="rainfall").copy()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(f"The target column 'rainfall' has been encoded into {len(np.unique(y_encoded))} classes.")

X_train, X_test, Y_train, Y_test = train_test_split(
    X, y_encoded, 
    test_size=.2, 
    random_state=RANDOM_STATE, 
    stratify=y_encoded
)

print("Data loaded and split.")
print(f"Training set shape: {X_train.shape}")
print(f"Test (validation) set shape: {X_test.shape}")
print("-" * 50)


# ==============================================================================
# 4. HYPERPARAMETER TUNING WITH OPTUNA
# ==============================================================================
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 2000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
    }
    model = CustomMulticlassGradientBoostingClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    score = cross_val_score(model, X_train, Y_train, n_jobs=-1, cv=cv, scoring='accuracy')
    return score.mean()

print("Starting hyperparameter optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
best_accuracy_cv = study.best_value

print("\n" + "="*50)
print("TUNING COMPLETED")
print("="*50)
print(f"\nBest mean accuracy during tuning (5-fold CV): {best_accuracy_cv:.4f}")
print(f"\nOptimal hyperparameter combination found:")
for param, value in best_params.items():
    print(f"  - {param}: {value}")
print("-" * 50)


# ==============================================================================
# 5. TRAINING AND FINAL EVALUATION OF THE OPTIMIZED MODEL
# ==============================================================================
print("Training and evaluating the final model with the optimal parameters...")

# Create the final model with the found parameters
final_model = CustomMulticlassGradientBoostingClassifier(**best_params)

# --- 5.1 Robust evaluation with Cross-Validation on the Training Set ---
print("\nRunning Cross-Validation on the final model...")
cv_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
final_scores = cross_val_score(final_model, X_train, Y_train, cv=cv_final, scoring='accuracy')

print(f"  -> Mean CV accuracy (5-fold): {final_scores.mean():.4f}")
print(f"  -> CV standard deviation: {final_scores.std():.4f}")

# --- 5.2 Train on the entire training set and evaluate on the Test Set ---
print("\nTraining the model on the entire training set...")
final_model.fit(X_train, Y_train)

print("Evaluating on the test (validation) set...")
y_pred_test = final_model.predict(X_test)
test_accuracy = accuracy_score(Y_test, y_pred_test)
print(f"  -> Final accuracy on the validation set: {test_accuracy:.4f}")
print("-" * 50)
print("Pipeline completed.")

🚀 Inizio: Caricamento e preparazione dei dati...
La colonna target 'rainfall' è stata codificata in 2 classi.
Dati caricati e suddivisi.
Dimensioni Training Set: (1752, 11)
Dimensioni Test (Validation) Set: (438, 11)
--------------------------------------------------
🤖 Avvio dell'ottimizzazione degli iperparametri...


Best trial: 21. Best value: 0.862449: 100%|██████████| 50/50 [01:38<00:00,  1.97s/it]



✅ TUNING COMPLETATO

🏆 Miglior accuracy media durante il tuning (CV 5-fold): 0.8624

⚙️ Combinazione di iperparametri ottimale trovata:
  - n_estimators: 150
  - learning_rate: 0.054398530041183925
  - max_depth: 2
--------------------------------------------------
🏋️ Training e valutazione del modello finale con i parametri ottimali...

Eseguendo la Cross-Validation sul modello finale...
  -> Accuracy media in CV (5-fold): 0.8624
  -> Deviazione Standard in CV: 0.0223

Training del modello sull'intero set di training...
Valutazione sul set di test (validation)...
  -> 🎯 Accuracy finale sul Validation Set: 0.8584
--------------------------------------------------
🏁 Pipeline completata.
